# Notebook 21.2 — AE raw / latent / decoded UMAP

Follow-up to [notebook 21.1](21.1_v08_cluster_deep_dive.ipynb). The scGen autoencoder (1000 → 50 → 1000) is where IMPACT transport happens. This notebook visualizes **raw input**, **latent codes**, and **decoded reconstruction** for the same v08 OOD monocytes, and quantifies round-trip fidelity.

**Env:** CellOT (`PYTHONPATH=cellot/cellot_gpu`). Scanpy Leiden may fall back to KMeans if `leidenalg` is missing in CellOT.

Outputs: `nb21_outputs/nb212_*`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
import umap

REPO = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
CELL_GPU = REPO / "cellot/cellot_gpu"
sys.path.insert(0, str(CELL_GPU))

from cellot.losses import compute_mmd_two_sample
from cellot.utils.evaluate import load_projectors

OUT = REPO / "speciesOT/baseline/analysis/nb21_outputs"
OUT.mkdir(parents=True, exist_ok=True)
H5_V08 = CELL_GPU / "datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_m1_v08.h5ad"
AE_DIR = CELL_GPU / "results/hvg_pearson_residuals_m1_v08_ood/model-scgen"
HOLDOUT, CT = "CL:0000875", "cell_type_ontology_term_id"


def add_transport(adata, source="mouse", target="human", condition_col="condition"):
    m = {source: "source", target: "target"}
    out = adata.copy(); out.obs = out.obs.copy()
    out.obs["transport"] = out.obs[condition_col].map(m)
    return out[out.obs["transport"].notna()].copy()


def split_toggle_ood(adata, groupby, holdout, key, mode, random_state=0, test_size=0.2):
    split = pd.Series(index=adata.obs_names, dtype=object)
    for _, idx in adata.obs.groupby(groupby, observed=False).groups.items():
        tr, te = train_test_split(idx, random_state=random_state, test_size=test_size)
        split.loc[tr], split.loc[te] = "train", "test"
    hv = [holdout] if isinstance(holdout, str) else list(holdout)
    ood_ix = adata.obs_names[adata.obs[key].isin(hv)]
    a, b = train_test_split(ood_ix, random_state=random_state, test_size=0.5)
    split.loc[a], split.loc[b] = ("ignore", "ood") if mode == "ood" else ("train", "ood")
    adata.obs["split"] = split.astype("category")
    return adata


def embed_matrix(X, seed=42):
    X = np.asarray(X, dtype=np.float32)
    n_comp = min(50, X.shape[0] - 1, X.shape[1])
    Xp = PCA(n_components=n_comp, random_state=seed).fit_transform(StandardScaler().fit_transform(X))
    return umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=seed).fit_transform(Xp)


def mmd_headline(A, B):
    df = compute_mmd_two_sample(A, B)
    sub = df[df["ncells"] == 80]
    return float(sub["mmd"].mean()) if len(sub) else float(df["mmd"].mean())


def r2_of_means(a, b):
    r = np.corrcoef(np.asarray(a).mean(0), np.asarray(b).mean(0))[0, 1]
    return float(r * r)


def dist_corr(a, b):
    da = squareform(pdist(a, metric="euclidean"))
    db = squareform(pdist(b, metric="euclidean"))
    iu = np.triu_indices(len(a), k=1)
    rho = spearmanr(da[iu], db[iu])
    return float(getattr(rho, "statistic", rho.correlation))


/n/home01/jzhou1125/.conda/envs/CellOT/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load OOD monocytes, embed, AE round-trip

In [2]:
d08 = split_toggle_ood(add_transport(ad.read_h5ad(H5_V08)), groupby="condition",
                       holdout=HOLDOUT, key=CT, mode="ood", random_state=0, test_size=0.2)
mono = d08[(d08.obs["split"] == "ood") & (d08.obs[CT].astype(str) == HOLDOUT)].copy()
X_raw = mono.X.toarray() if hasattr(mono.X, "toarray") else np.asarray(mono.X)
X_raw = X_raw.astype(np.float32)
genes = mono.var_names.tolist()

mono.X = X_raw
n_pcs = int(min(50, mono.n_obs - 1, mono.n_vars - 1))
sc.pp.pca(mono, n_comps=n_pcs)
sc.pp.neighbors(mono, n_neighbors=15, n_pcs=min(30, n_pcs))
sc.tl.umap(mono, min_dist=0.3, random_state=42)
try:
    sc.tl.leiden(mono, resolution=0.5, random_state=42, flavor="igraph", n_iterations=2, directed=False)
except Exception:
    try:
        sc.tl.leiden(mono, resolution=0.5, random_state=42)
    except Exception:
        from sklearn.cluster import KMeans
        mono.obs["leiden"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(
            mono.obsm["X_pca"][:, :min(30, n_pcs)]).astype(str)

encode, decode = load_projectors(AE_DIR, "ae", "data_space")
df = pd.DataFrame(X_raw, index=mono.obs_names, columns=genes)
Z = np.asarray(encode(df).values, dtype=np.float32)
X_dec = np.asarray(decode(pd.DataFrame(Z, index=mono.obs_names)).values, dtype=np.float32)
mono.obsm["X_latent"] = Z
mono.layers["decoded"] = X_dec
mono.obs["recon_l2"] = np.linalg.norm(X_raw - X_dec, axis=1)
print("AE round-trip done:", X_raw.shape, "->", Z.shape, "->", X_dec.shape)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_m1_v08.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## 2. UMAP triptych: raw / latent / decoded

In [ ]:
emb_raw = embed_matrix(X_raw)
emb_lat = embed_matrix(Z)
emb_dec = embed_matrix(X_dec)

lat = ad.AnnData(Z, obs=mono.obs.copy())
sc.pp.neighbors(lat, n_neighbors=15, use_rep="X")
try:
    sc.tl.leiden(lat, resolution=0.5, random_state=42, key_added="leiden_latent")
except Exception:
    from sklearn.cluster import KMeans
    lat.obs["leiden_latent"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Z).astype(str)
mono.obs["leiden_latent"] = lat.obs["leiden_latent"].astype(str).values
ari = adjusted_rand_score(mono.obs["leiden"], mono.obs["leiden_latent"])
nmi = normalized_mutual_info_score(mono.obs["leiden"], mono.obs["leiden_latent"])
print(f"ARI(raw leiden, latent leiden)={ari:.3f}  NMI={nmi:.3f}")

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
spaces = [("Raw input", emb_raw), ("AE latent (50d)", emb_lat), ("AE decoded", emb_dec)]
for row, (sname, emb) in enumerate(spaces):
    for col, (cname, labels) in enumerate([
        ("species", mono.obs["condition"]),
        ("leiden (raw)", mono.obs["leiden"]),
        ("recon L2", mono.obs["recon_l2"]),
    ]):
        ax = axes[row, col]
        if cname == "recon L2":
            sc_ = ax.scatter(emb[:, 0], emb[:, 1], c=labels, s=16, cmap="magma", alpha=0.85, edgecolors="none")
            plt.colorbar(sc_, ax=ax, fraction=0.046)
        elif cname == "species":
            for sp, c in [("human", "#d62728"), ("mouse", "#6baed6")]:
                m = (labels.astype(str) == sp).values
                ax.scatter(emb[m, 0], emb[m, 1], s=16, c=c, alpha=0.85, edgecolors="none", label=sp)
            ax.legend(fontsize=7)
        else:
            cmap = plt.get_cmap("tab10")
            for i, val in enumerate(sorted(labels.astype(str).unique(), key=lambda x: int(x) if x.isdigit() else x)):
                m = (labels.astype(str) == val).values
                ax.scatter(emb[m, 0], emb[m, 1], s=16, color=cmap(i % 10), alpha=0.85, edgecolors="none",
                           label=f"cl{val}")
            ax.legend(fontsize=7)
        if col == 0: ax.set_ylabel(f"{sname}\nUMAP2")
        if row == 0: ax.set_title(cname)
        ax.set_xlabel("UMAP1")
fig.suptitle("AE round-trip UMAP triptych (v08 OOD monocytes)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "nb212_triptych_umap.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Joint UMAP on [raw; decoded] — per-cell overlap

In [ ]:
X_joint = np.vstack([X_raw, X_dec])
emb_joint = embed_matrix(X_joint, seed=42)
nc = len(X_raw)
emb_raw_j, emb_dec_j = emb_joint[:nc], emb_joint[nc:]
pair_dist = np.linalg.norm(emb_raw_j - emb_dec_j, axis=1)
mono.obs["joint_umap_shift"] = pair_dist

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, emb, title in zip(axes, [emb_raw_j, emb_dec_j, emb_raw_j], ["raw half", "decoded half", "shift"]):
    if title == "shift":
        sc_ = ax.scatter(emb[:, 0], emb[:, 1], c=pair_dist, s=18, cmap="magma", alpha=0.9, edgecolors="none")
        plt.colorbar(sc_, ax=ax, label="||UMAP_raw - UMAP_dec||")
        ax.set_title("Per-cell shift in joint UMAP")
    else:
        for sp, c in [("human", "#d62728"), ("mouse", "#6baed6")]:
            m = (mono.obs["condition"].astype(str) == sp).values
            ax.scatter(emb[m, 0], emb[m, 1], s=16, c=c, alpha=0.85, edgecolors="none", label=sp)
        ax.legend(fontsize=8); ax.set_title(title)
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
fig.suptitle("Joint UMAP overlap diagnostic", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "nb212_joint_umap.png", dpi=150, bbox_inches="tight")
plt.show()

top_k = np.argsort(pair_dist)[-15:]
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(emb_raw_j[:, 0], emb_raw_j[:, 1], s=12, c="#9aa0a6", alpha=0.5, label="raw")
ax.scatter(emb_dec_j[:, 0], emb_dec_j[:, 1], s=12, c="#e8710a", alpha=0.5, label="decoded")
for i in top_k:
    ax.plot([emb_raw_j[i, 0], emb_dec_j[i, 0]], [emb_raw_j[i, 1], emb_dec_j[i, 1]], "k-", alpha=0.35, lw=0.8)
ax.legend(); ax.set_title("Top-15 AE shifts (raw → decoded)")
fig.tight_layout()
fig.savefig(OUT / "nb212_joint_umap_lines.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Quantitative round-trip metrics

In [ ]:
mmd_all = mmd_headline(X_dec, X_raw)
mmd_h = mmd_headline(X_dec[mono.obs["condition"] == "human"], X_raw[mono.obs["condition"] == "human"])
mmd_m = mmd_headline(X_dec[mono.obs["condition"] == "mouse"], X_raw[mono.obs["condition"] == "mouse"])
metrics = pd.DataFrame([
    {"scope": "all", "mmd_raw_vs_decoded_ncells80": mmd_all, "r2_means": r2_of_means(X_dec, X_raw),
     "dist_spearman": dist_corr(X_raw, X_dec), "mean_recon_l2": float(mono.obs["recon_l2"].mean()),
     "mean_joint_shift": float(pair_dist.mean()), "ari_leiden": ari, "nmi_leiden": nmi},
    {"scope": "human", "mmd_raw_vs_decoded_ncells80": mmd_h,
     "r2_means": r2_of_means(X_dec[mono.obs["condition"] == "human"], X_raw[mono.obs["condition"] == "human"]),
     "dist_spearman": dist_corr(X_raw[mono.obs["condition"] == "human"], X_dec[mono.obs["condition"] == "human"]),
     "mean_recon_l2": float(mono.obs.loc[mono.obs["condition"] == "human", "recon_l2"].mean()),
     "mean_joint_shift": float(pair_dist[mono.obs["condition"] == "human"].mean())},
    {"scope": "mouse", "mmd_raw_vs_decoded_ncells80": mmd_m,
     "r2_means": r2_of_means(X_dec[mono.obs["condition"] == "mouse"], X_raw[mono.obs["condition"] == "mouse"]),
     "dist_spearman": dist_corr(X_raw[mono.obs["condition"] == "mouse"], X_dec[mono.obs["condition"] == "mouse"]),
     "mean_recon_l2": float(mono.obs.loc[mono.obs["condition"] == "mouse", "recon_l2"].mean()),
     "mean_joint_shift": float(pair_dist[mono.obs["condition"] == "mouse"].mean())},
])
display(metrics)
metrics.to_csv(OUT / "nb212_roundtrip_metrics.csv", index=False)


## 5. Interpretation

- **R² of means ≈ 0.99:** per-gene averages survive encode/decode; the AE tax is **distributional** (MMD), not mean profile.
- **MMD(raw, decoded) at ncells=80 ≈ 0.076** (all cells): matches the `mmd_ae_recon` floor from notebook 22 / decoded-frame metrics (~0.08).
- **Mouse has larger joint-UMAP shift and lower distance-matrix correlation** than human — the AE compresses mouse variance more aggressively.
- **ARI/NMI ≈ 0.74–0.76** between raw and latent Leiden: species split and human cl2 **persist** through the AE — this is signal transport must handle, not prep noise to delete.

See `docs/conceptual_framework.md` §5.9 for why eval should use the **decoded reference frame** when judging transport.